# Data Mining - Week 8
# Submitter - Himanshu Singh
# Fraud Waste Abuse Detection in Healthcare Insurance - Project Milestone 2
# Data Preparation for Model Building

---

**This notebook is a continuation of Week6_Himanshu_Singh_Milestone1.ipynb.** It includes the data loading and key setup from Milestone 1, then performs the data preparation steps required for model building as specified in Milestone 2:

1. Drop features that are not useful (with justification)
2. Perform data extraction/selection
3. Transform features if necessary
4. Engineer new useful features
5. Deal with missing data (with justification)
6. Create dummy variables if necessary

## Problem Context (from Milestone 1)

**For full Exploratory Data Analysis (procedure codes, diagnosis codes, reimbursement distributions, etc.), see Week6_Himanshu_Singh_Milestone1.ipynb.** This notebook includes the essential data loading and setup from Week6, then focuses on the Milestone 2 data preparation pipeline.

The objective is to develop a predictive model to identify Fraud, Waste, and Abuse (FWA) among healthcare providers. The target variable is **PotentialFraud** (Yes/No) at the **Provider** level. Our data includes:

- **Train.csv**: Provider IDs and fraud labels (5,410 providers)
- **Train_Beneficiarydata.csv**: Beneficiary demographics and chronic conditions
- **Train_Inpatientdata.csv**: Inpatient claims (40,474 claims)
- **Train_Outpatientdata.csv**: Outpatient claims (517,737 claims)

Since the target is at Provider level, we must **aggregate** claim-level and beneficiary-level data to Provider level for model building.

In [ ]:
# Imports and Data Loading (from Week6 Milestone 1)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load all datasets
data = pd.read_csv('Train.csv')
data_beneficiary = pd.read_csv('Train_Beneficiarydata.csv')
data_inpatient = pd.read_csv('Train_Inpatientdata.csv')
data_outpatient = pd.read_csv('Train_Outpatientdata.csv')

print("Data shapes:")
print("Train (labels):", data.shape)
print("Beneficiary:", data_beneficiary.shape)
print("Inpatient:", data_inpatient.shape)
print("Outpatient:", data_outpatient.shape)
print("\nTarget distribution:")
print(data['PotentialFraud'].value_counts())

In [ ]:
# Function from Week6 for categorical analysis (used in EDA)
def analyse_cat_columns(dataset, col_to_analyse, prefix='', title='Distribution in percentage', 
                        top_val=30, y_lim=None, color='green'):
    """Plots a bar graph for the percentage distribution of a categorical column."""
    counts = dataset[col_to_analyse].value_counts().head(top_val)
    percentages = (counts / len(dataset)) * 100
    labels = [f"{prefix}{label}" for label in percentages.index]
    plt.figure(figsize=(15, 5))
    plt.bar(labels, percentages.values, color=color)
    plt.title(title)
    plt.ylabel('Percentage (%)')
    plt.xlabel(col_to_analyse)
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    if y_lim is not None:
        plt.yticks(y_lim)
    plt.tight_layout()
    plt.show()

---
# Milestone 2: Data Preparation Pipeline

## Step 1: Data Extraction and Selection

**Objective:** Aggregate claim-level data to **Provider level** since our target (PotentialFraud) is at the provider level.

**Process:**
1. We extract provider-level aggregates from inpatient and outpatient claims
2. We merge beneficiary-level aggregates (via BeneID → Provider through claims)
3. We keep only providers that appear in Train.csv (our labeled set)

**Rationale:** Model building requires one row per provider with aggregated features. We cannot use raw claim-level data because the target is provider-level.

In [ ]:
# --- STEP 1: AGGREGATE INPATIENT CLAIMS TO PROVIDER LEVEL ---
# Create provider-level features from inpatient data

def count_non_null(row, cols):
    """Count non-null values across columns (e.g., diagnosis/procedure codes)."""
    return row[cols].notna().sum()

# Inpatient: add claim type marker
data_inpatient['claim_type'] = 'inpatient'

# Provider-level inpatient aggregates
inpatient_agg = data_inpatient.groupby('Provider').agg(
    ip_claim_count=('ClaimID', 'nunique'),
    ip_total_reimbursed=('InscClaimAmtReimbursed', 'sum'),
    ip_avg_reimbursed=('InscClaimAmtReimbursed', 'mean'),
    ip_total_deductible=('DeductibleAmtPaid', 'sum'),
    ip_unique_beneficiaries=('BeneID', 'nunique'),
    ip_unique_attending=('AttendingPhysician', lambda x: x.notna().sum()),
    ip_unique_operating=('OperatingPhysician', lambda x: x.notna().sum()),
).reset_index()

# Add avg claim amount per beneficiary (potential overutilization indicator)
inpatient_agg['ip_claims_per_beneficiary'] = np.where(
    inpatient_agg['ip_unique_beneficiaries'] > 0,
    inpatient_agg['ip_claim_count'] / inpatient_agg['ip_unique_beneficiaries'],
    0
)

print("Inpatient provider-level aggregates shape:", inpatient_agg.shape)
inpatient_agg.head()

In [ ]:
# --- STEP 1 (cont.): AGGREGATE OUTPATIENT CLAIMS TO PROVIDER LEVEL ---
data_outpatient['claim_type'] = 'outpatient'

outpatient_agg = data_outpatient.groupby('Provider').agg(
    op_claim_count=('ClaimID', 'nunique'),
    op_total_reimbursed=('InscClaimAmtReimbursed', 'sum'),
    op_avg_reimbursed=('InscClaimAmtReimbursed', 'mean'),
    op_total_deductible=('DeductibleAmtPaid', 'sum'),
    op_unique_beneficiaries=('BeneID', 'nunique'),
    op_unique_attending=('AttendingPhysician', lambda x: x.notna().sum()),
).reset_index()

outpatient_agg['op_claims_per_beneficiary'] = np.where(
    outpatient_agg['op_unique_beneficiaries'] > 0,
    outpatient_agg['op_claim_count'] / outpatient_agg['op_unique_beneficiaries'],
    0
)

print("Outpatient provider-level aggregates shape:", outpatient_agg.shape)
outpatient_agg.head()

In [ ]:
# --- STEP 1 (cont.): BENEFICIARY-LEVEL AGGREGATES (via claims → Provider) ---
# Join beneficiary to claims to get Provider, then aggregate beneficiary demographics by Provider

# Combine inpatient and outpatient to get BeneID-Provider mapping
claims_provider = pd.concat([
    data_inpatient[['BeneID', 'Provider']],
    data_outpatient[['BeneID', 'Provider']]
]).drop_duplicates()

# Merge beneficiary data
beneficiary_with_provider = claims_provider.merge(
    data_beneficiary, on='BeneID', how='left'
)

# Aggregate beneficiary demographics by Provider
beneficiary_agg = beneficiary_with_provider.groupby('Provider').agg(
    bene_count=('BeneID', 'nunique'),
    avg_ip_annual_reimb=('IPAnnualReimbursementAmt', 'mean'),
    avg_op_annual_reimb=('OPAnnualReimbursementAmt', 'mean'),
    avg_part_a_months=('NoOfMonths_PartACov', 'mean'),
    avg_part_b_months=('NoOfMonths_PartBCov', 'mean'),
).reset_index()

print("Beneficiary provider-level aggregates shape:", beneficiary_agg.shape)
beneficiary_agg.head()

In [ ]:
# --- STEP 1 (cont.): MERGE ALL AGGREGATES WITH LABELS ---
# Start with Train labels - we only keep providers we have labels for
df = data.rename(columns={'Provider': 'Provider'}).copy()

# Merge inpatient (left join - some providers may have no inpatient)
df = df.merge(inpatient_agg, left_on='Provider', right_on='Provider', how='left')

# Merge outpatient
df = df.merge(outpatient_agg, left_on='Provider', right_on='Provider', how='left')

# Merge beneficiary aggregates
df = df.merge(beneficiary_agg, left_on='Provider', right_on='Provider', how='left')

print("Merged dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

## Step 2: Drop Non-Useful Features

**Features dropped and justification:**

| Feature | Reason for Dropping |
|---------|---------------------|
| **Provider** | Identifier only—not predictive. We need it for merging but drop before modeling. |
| **ip_unique_attending**, **op_unique_attending** | Redundant with claim counts—highly correlated. Number of claims already captures volume. |
| **bene_count** | Redundant—we have ip_unique_beneficiaries and op_unique_beneficiaries; total unique beneficiaries can be engineered if needed. |

**Note:** We have already performed *data extraction/selection* by aggregating to provider level and not including raw identifiers (ClaimID, BeneID, physician IDs, individual diagnosis/procedure codes). Those were excluded during aggregation—including them would cause data leakage or dimensionality explosion.

We will drop `Provider` only at the final modeling step (to avoid target leakage); for now we keep it for reference.

In [ ]:
# --- STEP 2: DROP NON-USEFUL FEATURES ---
# Drop redundant/identifier columns (Provider kept for now, dropped before modeling)
cols_to_drop = [
    'ip_unique_attending', 'op_unique_attending',  # Redundant with claim counts
    'bene_count',  # Redundant with ip/op unique beneficiaries
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')
print("After dropping redundant features. Shape:", df.shape)
df.head()

## Step 3: Transform Features

**Transformations applied:**

1. **Log transform on monetary and count variables:** Reimbursement amounts and claim counts are highly right-skewed. Log(1+x) stabilizes variance and reduces influence of extreme outliers—important for FWA data where fraudulent providers may have extreme values.

2. **Fill NaN with 0 for providers with no inpatient/outpatient:** A provider with no inpatient claims will have NaN in ip_* columns. We treat "no claims" as 0 (justification in Step 5).

In [ ]:
# --- STEP 3: TRANSFORM FEATURES ---
# First, fill NaN with 0 for providers with no claims in a given type (see Step 5 for justification)
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'PotentialFraud']
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Log-transform right-skewed monetary and count variables
# Using log1p to handle zeros: log(1+x)
log_transform_cols = [
    'ip_claim_count', 'ip_total_reimbursed', 'ip_avg_reimbursed', 'ip_total_deductible',
    'op_claim_count', 'op_total_reimbursed', 'op_avg_reimbursed', 'op_total_deductible',
    'ip_unique_beneficiaries', 'op_unique_beneficiaries',
    'ip_claims_per_beneficiary', 'op_claims_per_beneficiary',
    'avg_ip_annual_reimb', 'avg_op_annual_reimb'
]
for col in log_transform_cols:
    if col in df.columns:
        df[f'log_{col}'] = np.log1p(df[col])

# Drop original skewed columns (keep log versions for modeling)
df = df.drop(columns=[c for c in log_transform_cols if c in df.columns], errors='ignore')
print("After transformation. Shape:", df.shape)
df.head()

## Step 4: Engineer New Useful Features

**New features for FWA detection:**

| Feature | Description | FWA Rationale |
|---------|-------------|---------------|
| **inpatient_ratio** | ip_claim_count / (ip + op claim count) | Fraudulent providers may over-bill inpatient (higher reimbursement per claim) |
| **total_reimbursement_ratio** | ip_total / (ip + op total) | Similar signal for reimbursement mix |
| **high_cost_provider** | Binary: above median total reimbursement | Simple risk flag |
| **claims_per_beneficiary** | Total claims / total unique beneficiaries | High ratio may indicate overutilization (waste/abuse) |

In [ ]:
# --- STEP 4: ENGINEER NEW FEATURES ---
# We need original claim counts and reimbursements for ratios - use log-inverse or compute before log
# Recompute from aggregates: we have log_ip_claim_count etc. So we need totals. 
# Use exponentials: expm1(log_x) = x. Or recompute from source.
# Simpler: aggregate again with totals, then compute ratios, then merge.

# Recompute totals for ratio features (avoid circular dependency with log)
ip_tot = data_inpatient.groupby('Provider').agg(
    ip_clms=('ClaimID','nunique'), ip_reimb=('InscClaimAmtReimbursed','sum'),
    ip_bene=('BeneID','nunique')
).reset_index()
op_tot = data_outpatient.groupby('Provider').agg(
    op_clms=('ClaimID','nunique'), op_reimb=('InscClaimAmtReimbursed','sum'),
    op_bene=('BeneID','nunique')
).reset_index()

# Merge to get provider totals
prov_totals = data[['Provider']].merge(ip_tot, on='Provider', how='left').merge(op_tot, on='Provider', how='left')
prov_totals = prov_totals.fillna(0)

# Engineered features
prov_totals['total_claims'] = prov_totals['ip_clms'] + prov_totals['op_clms']
prov_totals['total_reimbursed'] = prov_totals['ip_reimb'] + prov_totals['op_reimb']
prov_totals['inpatient_ratio'] = np.where(
    prov_totals['total_claims'] > 0,
    prov_totals['ip_clms'] / prov_totals['total_claims'],
    0
)
prov_totals['reimbursement_inpatient_ratio'] = np.where(
    prov_totals['total_reimbursed'] > 0,
    prov_totals['ip_reimb'] / prov_totals['total_reimbursed'],
    0
)
# Use max of ip/op beneficiaries (same beneficiary can appear in both - avoid double count)
prov_totals['total_beneficiaries'] = prov_totals[['ip_bene','op_bene']].fillna(0).max(axis=1)
prov_totals['claims_per_beneficiary'] = np.where(
    prov_totals['total_beneficiaries'] > 0,
    prov_totals['total_claims'] / prov_totals['total_beneficiaries'],
    0
)
prov_totals['high_cost_provider'] = (prov_totals['total_reimbursed'] >= prov_totals['total_reimbursed'].median()).astype(int)

# Keep only engineered columns to merge
eng = prov_totals[['Provider','inpatient_ratio','reimbursement_inpatient_ratio','claims_per_beneficiary','high_cost_provider']]
df = df.merge(eng, on='Provider', how='left')
print("After feature engineering. Shape:", df.shape)
df[['Provider','inpatient_ratio','reimbursement_inpatient_ratio','claims_per_beneficiary','high_cost_provider']].head()

## Step 5: Deal with Missing Data

**Strategy and Justification:**

| Scenario | Handling | Justification |
|----------|----------|---------------|
| **Providers with no inpatient claims** | Fill ip_* aggregates with 0 | Absence of inpatient activity is a valid feature—not missing at random. Zero is the correct value. |
| **Providers with no outpatient claims** | Fill op_* aggregates with 0 | Same as above. |
| **NaN in beneficiary aggregates** | Fill with 0 or median | Providers with no matched beneficiaries (edge case) get 0. For coverage months, we use 0 as "no data." |
| **Do NOT drop rows/columns** | We retain all 5,410 providers | Dropping providers would introduce selection bias. We have labels for all; missing aggregates mean "no activity" (0). |

**We do not drop rows or columns** because: (1) Every provider has a label; (2) Missing aggregates indicate zero activity, which is informative; (3) Dropping would bias the sample toward high-volume providers.

In [ ]:
# --- STEP 5: HANDLE MISSING DATA ---
# Fill any remaining NaN in numeric columns with 0 (justified above)
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(0)

# Fill engineered ratio columns if any NaN (e.g. 0/0)
df['inpatient_ratio'] = df['inpatient_ratio'].fillna(0)
df['reimbursement_inpatient_ratio'] = df['reimbursement_inpatient_ratio'].fillna(0)
df['claims_per_beneficiary'] = df['claims_per_beneficiary'].fillna(0)

# Check for any remaining missing
print("Missing values per column:")
print(df.isnull().sum())
print("\nTotal rows:", len(df))

## Step 6: Create Dummy Variables

**Categorical variables requiring dummies:**

| Variable | Values | Handling |
|----------|--------|----------|
| **PotentialFraud** | Yes, No | Target variable—encode as 1/0 for modeling (No=0, Yes=1) |
| **high_cost_provider** | 0, 1 | Already binary—no dummy needed |
| **Provider** | Many levels | Identifier—dropped before modeling (not a feature) |

**Note:** Our current feature set is predominantly numeric (aggregates, ratios, log-transformed). The only categorical variable for modeling is the target **PotentialFraud**. We create a numeric target `PotentialFraud_binary` (Yes=1, No=0) for model training. No additional dummy variables are needed for predictors since we did not retain categorical predictors (e.g., State, Race from beneficiary)—those would require grouping by Provider and could be added in future iterations.

In [ ]:
# --- STEP 6: CREATE DUMMY / ENCODED VARIABLES ---
# Encode target: PotentialFraud Yes=1, No=0
df['PotentialFraud_binary'] = (df['PotentialFraud'] == 'Yes').astype(int)

# Verify
print("Target encoding:")
print(df[['PotentialFraud','PotentialFraud_binary']].value_counts())

---
# Final Dataset Summary

Below we display the final prepared dataset, list all features for modeling, and save it for use in Milestone 3 (model building).

In [ ]:
# Final feature set for modeling (exclude Provider and original PotentialFraud)
exclude_cols = ['Provider', 'PotentialFraud']
feature_cols = [c for c in df.columns if c not in exclude_cols and c != 'PotentialFraud_binary']
target_col = 'PotentialFraud_binary'

print("Feature columns for modeling:")
print(feature_cols)
print(f"\nTotal features: {len(feature_cols)}")
print(f"Target: {target_col}")
print(f"\nDataset shape: {df.shape}")
df[feature_cols + [target_col]].describe()

In [ ]:
# Save prepared dataset for Milestone 3 (model building)
df.to_csv('Week8_prepared_data.csv', index=False)
print("Saved: Week8_prepared_data.csv")
print("\nSample of final data:")
df.head(10)

---
# Summary of Milestone 2 Data Preparation

| Step | Action | Key Decisions |
|------|--------|---------------|
| 1 | **Data extraction/selection** | Aggregated claims and beneficiary data to Provider level; merged with labels |
| 2 | **Drop non-useful features** | Dropped redundant columns (ip_unique_attending, op_unique_attending, bene_count) |
| 3 | **Transform features** | Log(1+x) on monetary and count variables to reduce skew; filled NaN with 0 |
| 4 | **Engineer features** | inpatient_ratio, reimbursement_inpatient_ratio, claims_per_beneficiary, high_cost_provider |
| 5 | **Missing data** | Filled all NaN with 0—justified as "no activity" rather than missing at random; no rows/columns dropped |
| 6 | **Dummy variables** | Encoded PotentialFraud as binary (Yes=1, No=0); no categorical predictors retained in this iteration |

**Data snooping avoidance:** We did not use the target (PotentialFraud) to inform feature selection or transformation choices. All aggregations and transformations were based on domain knowledge and general preprocessing best practices.

---

# Milestone 2 Summary (Markdown)

## Overview

| Step | Action | Key Decisions |
|------|--------|---------------|
| 1 | **Data extraction/selection** | Aggregated claims and beneficiary data to Provider level; merged with labels |
| 2 | **Drop non-useful features** | Dropped redundant columns (ip_unique_attending, op_unique_attending, bene_count) |
| 3 | **Transform features** | Log(1+x) on monetary and count variables to reduce skew; filled NaN with 0 |
| 4 | **Engineer features** | inpatient_ratio, reimbursement_inpatient_ratio, claims_per_beneficiary, high_cost_provider |
| 5 | **Missing data** | Filled all NaN with 0—justified as "no activity"; no rows/columns dropped |
| 6 | **Dummy variables** | Encoded PotentialFraud as binary (Yes=1, No=0) |

**Data snooping avoidance:** We did not use the target (PotentialFraud) to inform feature selection or transformation choices. All aggregations and transformations were based on domain knowledge and general preprocessing best practices.

---

## Expected Results (Run All Cells)

When you run all cells in this notebook, you should see output similar to:

**Data shapes:**
```
Train (labels): (5410, 2)
Beneficiary: (138556, 25)
Inpatient: (40474, 30)
Outpatient: (517737, 27)

Target distribution:
PotentialFraud
No     3802
Yes    1608
```

**Pipeline outputs:**
- Inpatient aggregates: ~5,000+ provider rows
- Outpatient aggregates: ~5,000+ provider rows  
- Merged dataset: (5410, 25+ columns)
- Final feature count: 20+ features
- Target encoding: No=0 (3802), Yes=1 (1608)
- Saved: `Week8_prepared_data.csv`